# Assignment No. 9 — NLP Text Processing Techniques

**Given sentences:**

1. *"The children are playing happily in the gardens."*
2. *"I am running and walking every morning."*
3. *"The organization organized an event."*

## Setup

In [2]:
# If NLTK is missing, run once:  !pip install nltk
import re
import string
from datetime import datetime
import pandas as pd
import nltk

for pkg in ["punkt", "punkt_tab", "wordnet", "stopwords", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.stem import PorterStemmer, SnowballStemmer, WordNetLemmatizer
from nltk import pos_tag
from nltk.corpus import wordnet, stopwords

sentences = [
    "The children are playing happily in the gardens.",
    "I am running and walking every morning.",
    "The organization organized an event.",
]
paragraph = " ".join(sentences)
print(paragraph)

The children are playing happily in the gardens. I am running and walking every morning. The organization organized an event.


## Question 1: Word and sentence tokenization

**Sentence tokenization** splits a text into sentences. **Word tokenization** splits a text into words and punctuation marks (tokens).

In [3]:
# Sentence tokenization (applied to the whole paragraph)
sent_tokens = sent_tokenize(paragraph)
print("Number of sentences:", len(sent_tokens))
for i, s in enumerate(sent_tokens, 1):
    print(f"  {i}. {s}")

Number of sentences: 3
  1. The children are playing happily in the gardens.
  2. I am running and walking every morning.
  3. The organization organized an event.


In [4]:
# Word tokenization (applied to each sentence)
word_tokens = {}
for s in sentences:
    word_tokens[s] = word_tokenize(s)
    print(s)
    print("  Tokens :", word_tokens[s])
    print("  Count  :", len(word_tokens[s]), "(the full stop counts as a token)")
    print()

The children are playing happily in the gardens.
  Tokens : ['The', 'children', 'are', 'playing', 'happily', 'in', 'the', 'gardens', '.']
  Count  : 9 (the full stop counts as a token)

I am running and walking every morning.
  Tokens : ['I', 'am', 'running', 'and', 'walking', 'every', 'morning', '.']
  Count  : 8 (the full stop counts as a token)

The organization organized an event.
  Tokens : ['The', 'organization', 'organized', 'an', 'event', '.']
  Count  : 6 (the full stop counts as a token)



## Question 2: Difference between word segmentation and tokenization

| | **Word segmentation** | **Tokenization** |
|---|---|---|
| **Meaning** | Finding the boundaries between words when the text does **not** clearly show them | Splitting text into **tokens** (words, numbers, punctuation, symbols) |
| **Main problem** | Deciding *where* one word ends and the next begins | Deciding what counts as a token (e.g. is `don't` one token or two? is `.` a token?) |
| **Typical languages / text** | Chinese, Japanese, Thai (written without spaces), and joined text such as hashtags or URLs: `#naturallanguageprocessing` | Languages like English and Urdu, where spaces and punctuation already mark most boundaries |
| **Difficulty** | Hard: needs a dictionary or a statistical model, and the text can often be split in more than one way | Easier: rules based on spaces and punctuation are usually enough |
| **Example** | `ilovenaturallanguageprocessing` → `i love natural language processing` | `"I love NLP."` → `["I", "love", "NLP", "."]` |

Word segmentation is therefore a special, harder part of tokenization. For English the boundaries are already marked by spaces, so tokenization is mostly about handling punctuation and special cases, while segmentation is needed when spaces are missing.

In [5]:
# Demonstration with a simple dictionary-based segmenter (maximum matching)
dictionary = {"i", "love", "natural", "language", "processing", "nlp", "the", "children", "play", "in", "garden"}

def segment(text, vocab, max_len=12):
    words, i = [], 0
    while i < len(text):
        for j in range(min(len(text), i + max_len), i, -1):     # try the longest match first
            if text[i:j] in vocab:
                words.append(text[i:j])
                i = j
                break
        else:                                                   # unknown character
            words.append(text[i])
            i += 1
    return words

joined = "ilovenaturallanguageprocessing"
print("Joined text        :", joined)
print("Word segmentation  :", segment(joined, dictionary))
print()
print("Normal text        : I love natural language processing.")
print("Tokenization       :", word_tokenize("I love natural language processing."))

Joined text        : ilovenaturallanguageprocessing
Word segmentation  : ['i', 'love', 'natural', 'language', 'processing']

Normal text        : I love natural language processing.
Tokenization       : ['I', 'love', 'natural', 'language', 'processing', '.']


## Question 3: Stemming

**Stemming** cuts the ends of words (suffixes) using fixed rules to reduce them to a *stem*. The stem does not have to be a real word. The Porter stemmer and the Snowball stemmer are applied.

In [6]:
porter = PorterStemmer()
snowball = SnowballStemmer("english")

for s in sentences:
    toks = [t for t in word_tokenize(s) if t not in string.punctuation]
    print(s)
    print("  Porter  :", [porter.stem(t) for t in toks])
    print("  Snowball:", [snowball.stem(t) for t in toks])
    print()

The children are playing happily in the gardens.
  Porter  : ['the', 'children', 'are', 'play', 'happili', 'in', 'the', 'garden']
  Snowball: ['the', 'children', 'are', 'play', 'happili', 'in', 'the', 'garden']

I am running and walking every morning.
  Porter  : ['i', 'am', 'run', 'and', 'walk', 'everi', 'morn']
  Snowball: ['i', 'am', 'run', 'and', 'walk', 'everi', 'morn']

The organization organized an event.
  Porter  : ['the', 'organ', 'organ', 'an', 'event']
  Snowball: ['the', 'organ', 'organ', 'an', 'event']



In [7]:
stem_rows = []
for s in sentences:
    for t in word_tokenize(s):
        if t not in string.punctuation:
            stem_rows.append({"Word": t, "Porter stem": porter.stem(t), "Snowball stem": snowball.stem(t)})
stem_table = pd.DataFrame(stem_rows)
stem_table

,Word,Porter stem,Snowball stem
0,The,the,the
1,children,children,children
2,are,are,are
3,playing,play,play
4,happily,happili,happili
5,in,in,in
6,the,the,the
7,gardens,garden,garden
8,I,i,i
9,am,am,am


## Question 4: Text normalization

**Text normalization** converts text into one standard, consistent form so that different spellings of the same word are treated as the same. The steps used here are:

1. Convert to lower case
2. Tokenize
3. Remove punctuation
4. **Lemmatize**: convert each word to its dictionary form (lemma) using its part of speech (the verb *are* becomes *be*, the noun *children* becomes *child*)

In [8]:
lemmatizer = WordNetLemmatizer()

def to_wordnet_pos(tag):
    if tag.startswith("J"): return wordnet.ADJ
    if tag.startswith("V"): return wordnet.VERB
    if tag.startswith("R"): return wordnet.ADV
    return wordnet.NOUN

def normalize(sentence):
    tokens = word_tokenize(sentence.lower())                         # 1 + 2
    tokens = [t for t in tokens if t not in string.punctuation]      # 3
    tagged = pos_tag(tokens)
    return [lemmatizer.lemmatize(w, to_wordnet_pos(tag)) for w, tag in tagged]   # 4

for s in sentences:
    print("Original  :", s)
    print("Normalized:", normalize(s))
    print("As text   :", " ".join(normalize(s)))
    print()

Original  : The children are playing happily in the gardens.
Normalized: ['the', 'child', 'be', 'play', 'happily', 'in', 'the', 'garden']
As text   : the child be play happily in the garden

Original  : I am running and walking every morning.
Normalized: ['i', 'be', 'run', 'and', 'walk', 'every', 'morning']
As text   : i be run and walk every morning

Original  : The organization organized an event.
Normalized: ['the', 'organization', 'organize', 'an', 'event']
As text   : the organization organize an event



## Question 5: Regular expressions for emails, phone numbers and dates

The three given sentences do not contain any emails, phone numbers or dates, so the expressions are tested on a sample text that contains valid examples and some invalid ones that should **not** match.

In [10]:
sample_text = """
Contact us at ali.khan@netsol.com or support_team@my-company.co.uk for help.
Wrong emails: user@@mail..com, @nouser.com, plain.text, name@domain

Phone numbers: +92 300 1234567, 0300-1234567, 03001234567, +1 (555) 123-4567, 555-123-4567.
Not a phone number: order id 123456789012, year 2024.

Dates: 25/12/2024, 05-03-2023, 2024-12-25, December 25, 2024, 25 Dec 2024, 1st January 2025.
Not a date: 99/99/9999 and the version 1.2.3.
"""
print(sample_text)


Contact us at ali.khan@netsol.com or support_team@my-company.co.uk for help.
Wrong emails: user@@mail..com, @nouser.com, plain.text, name@domain

Phone numbers: +92 300 1234567, 0300-1234567, 03001234567, +1 (555) 123-4567, 555-123-4567.
Not a phone number: order id 123456789012, year 2024.

Dates: 25/12/2024, 05-03-2023, 2024-12-25, December 25, 2024, 25 Dec 2024, 1st January 2025.
Not a date: 99/99/9999 and the version 1.2.3.



### 5. a. Email addresses

```
\b[A-Za-z0-9._%+-]+@[A-Za-z0-9-]+(?:\.[A-Za-z0-9-]+)*\.[A-Za-z]{2,}\b
```

| Part | Meaning |
|---|---|
| `[A-Za-z0-9._%+-]+` | Local part: one or more letters, digits or the symbols `. _ % + -` |
| `@` | The `@` symbol |
| `[A-Za-z0-9-]+(?:\.[A-Za-z0-9-]+)*` | Domain name, optionally with sub-domains (`my-company`) |
| `\.[A-Za-z]{2,}` | Dot and a top-level domain of at least 2 letters (`com`, `co.uk`, `org`) |

In [11]:
email_pattern = r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9-]+(?:\.[A-Za-z0-9-]+)*\.[A-Za-z]{2,}\b"
emails = re.findall(email_pattern, sample_text)
print("Emails found:", emails)

Emails found: ['ali.khan@netsol.com', 'support_team@my-company.co.uk']


### 5. b. Phone numbers

```
(?<!\d)(?:\+\d{1,3}[\s-]?)?(?:\(\d{3}\)\s?|\d{3,4}[\s-]?)\d{3}[\s-]?\d{4}(?!\d)
```

| Part | Meaning |
|---|---|
| `(?<!\d)` and `(?!\d)` | The number must not be part of a longer string of digits |
| `(?:\+\d{1,3}[\s-]?)?` | Optional country code such as `+92` or `+1` |
| `(?:\(\d{3}\)\s?\|\d{3,4}[\s-]?)` | Area or operator code: `(555)` or `0300` / `555`, with an optional space or hyphen |
| `\d{3}[\s-]?\d{4}` | The remaining 7 digits, optionally split by a space or hyphen |

In [17]:
phone_pattern = r"(?<!\d)(?:\+\d{1,3}[\s-]?)?(?:\(\d{3}\)\s?|\d{3,4}[\s-]?)\d{3}[\s-]?\d{4}(?!\d)"
phones = [m.group().strip() for m in re.finditer(phone_pattern, sample_text)]
print("Phone numbers found:")
for p in phones:
    print("  ", p)

Phone numbers found:
   +92 300 1234567
   0300-1234567
   03001234567
   +1 (555) 123-4567
   555-123-4567


### 5. c. Dates

Four common formats are covered: `dd/mm/yyyy` or `dd-mm-yyyy`, `yyyy-mm-dd`, `Month dd, yyyy` and `dd Month yyyy`. The day is limited to 01–31 and the month to 01–12.

In [12]:
months = r"(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:t(?:ember)?)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)"
day    = r"(?:0?[1-9]|[12]\d|3[01])"
month  = r"(?:0?[1-9]|1[0-2])"

date_pattern = (
    rf"\b{day}[/-]{month}[/-]\d{{4}}\b"                          # 25/12/2024, 05-03-2023
    rf"|\b\d{{4}}-(?:0[1-9]|1[0-2])-(?:0[1-9]|[12]\d|3[01])\b"   # 2024-12-25
    rf"|\b{months}\.? {day}(?:st|nd|rd|th)?,? \d{{4}}\b"        # December 25, 2024
    rf"|\b{day}(?:st|nd|rd|th)? {months}\.?,? \d{{4}}\b"        # 25 Dec 2024, 1st January 2025
)
dates = [m.group() for m in re.finditer(date_pattern, sample_text)]
print("Dates found:")
for d in dates:
    print("  ", d)

Dates found:
   25/12/2024
   05-03-2023
   2024-12-25
   December 25, 2024
   25 Dec 2024
   1st January 2025


In [15]:
# A regular expression checks the FORMAT only. A real calendar check can be added with datetime:
for d in ["25/12/2024", "31/02/2024"]:
    try:
        datetime.strptime(d, "%d/%m/%Y")
        print(f"{d}: valid date")
    except ValueError:
        print(f"{d}: matches the pattern but is NOT a real date (February has no 31st)")

25/12/2024: valid date
31/02/2024: matches the pattern but is NOT a real date (February has no 31st)


In [18]:
# Summary
pd.DataFrame({
    "Type": ["Emails", "Phone numbers", "Dates"],
    "Found": [len(emails), len(phones), len(dates)],
    "Matches": [emails, phones, dates],
})

,Type,Found,Matches
0,Emails,2,"[ali.khan@netsol.com, support_team@my-company...."
1,Phone numbers,5,"[+92 300 1234567, 0300-1234567, 03001234567, +..."
2,Dates,6,"[25/12/2024, 05-03-2023, 2024-12-25, December ..."


## Question 6: Compare the results of stemming and normalization

In [19]:
rows = []
for s in sentences:
    toks = [t for t in word_tokenize(s) if t not in string.punctuation]
    lemmas = normalize(s)
    for t, l in zip(toks, lemmas):
        rows.append({"Original": t, "Stem (Porter)": porter.stem(t), "Normalized (lower + lemma)": l})
compare = pd.DataFrame(rows)
compare

,Original,Stem (Porter),Normalized (lower + lemma)
0,The,the,the
1,children,children,child
2,are,are,be
3,playing,play,play
4,happily,happili,happily
5,in,in,in
6,the,the,the
7,gardens,garden,garden
8,I,i,i
9,am,am,be


In [20]:
all_words = [w.lower() for w in compare["Original"]]
print("Unique words      :", len(set(all_words)))
print("Unique stems      :", len(set(compare["Stem (Porter)"])))
print("Unique lemmas     :", len(set(compare["Normalized (lower + lemma)"])))
print()

stop_words = set(stopwords.words("english"))
not_words = [s for s in set(compare["Stem (Porter)"]) if s not in stop_words and not wordnet.synsets(s)]
print("Stems that are not dictionary words (stopwords ignored):", sorted(not_words))
print()
print("Words reduced to the same stem:")
print(compare.groupby("Stem (Porter)")["Original"].apply(list)[lambda x: x.str.len() > 1].to_string())

Unique words      : 18
Unique stems      : 17
Unique lemmas     : 17

Stems that are not dictionary words (stopwords ignored): ['everi', 'happili']

Words reduced to the same stem:
Stem (Porter)
organ    [organization, organized]
the                [The, the, The]


## Comparison: stemming vs normalization

| Word | Stemming (Porter) | Normalization (lower case + lemma) | Better result |
|---|---|---|---|
| children | children (unchanged) | **child** | Normalization: handles irregular plurals |
| are, am | are, am (unchanged) | **be**, **be** | Normalization: groups irregular verb forms |
| happily | happili (not a word) | happily | Normalization: keeps a real word |
| every | everi (not a word) | every | Normalization |
| morning | morn | morning | Normalization: keeps the original meaning |
| organization | organ | organization | Normalization: *organ* is a different word (a body part) |
| organized | organ | organize | Normalization |
| playing, running, walking, gardens | play, run, walk, garden | play, run, walk, garden | Same result |

| Point | Stemming | Normalization (with lemmatization) |
|---|---|---|
| **Method** | Cuts word endings with fixed rules | Uses a dictionary and the part of speech of each word |
| **Output** | Stem, often **not a real word** (`happili`, `everi`) | A real dictionary word (lemma) |
| **Speed** | Faster, simple | Slower: needs a dictionary and POS tagging |
| **Irregular forms** | Not handled (`children`, `are`, `am`) | Handled (`child`, `be`) |
| **Errors** | **Over-stemming**: `organization` and `organized` both become `organ`, which loses the meaning. **Under-stemming**: `children` is not reduced to `child` | Rarely produces wrong words, but depends on correct POS tags |
| **Other steps** | Only shortens words | Also includes lower-casing and punctuation removal, so the whole text becomes consistent |

**Conclusion:** Both techniques reduce the number of different word forms (the 18 unique words became 17 unique stems and 17 unique lemmas), but in different ways. Stemming merged *organization* and *organized* into one stem that has a different meaning, and it could not link *are* and *am*. Normalization correctly merged *are* and *am* into *be*, changed *children* to *child*, and kept every word readable. **Stemming** is a good choice when speed matters and exact words do not, for example in search engines. **Normalization with lemmatization** is better when the meaning of the words must be preserved, for example in chatbots, text classification and translation.